In [15]:
import os
import shutil
import random
import re

##Define Dataset Paths##

In [16]:
base_dir = "data/final/ddd"
output_dir = "data/split_new/ddd"

labels = ["drowsy", "non_drowsy"]

print("Base Folder:", base_dir)
print("Output Folder:", output_dir)

Base Folder: data/final/ddd
Output Folder: data/split_new/ddd


In [ ]:
##checking the original folders
for label in labels:
    path = os.path.join(base_dir, label)
    files = os.listdir(path)
    
    print(label, "contains", len(files), "images")

drowsy contains 22157 images
non_drowsy contains 19307 images


In [ ]:
##extract the prefix
def get_prefix(filename):
    match = re.match(r"[A-Za-z]+", filename)
    return match.group()
    
print(get_prefix("A001.png"))
print(get_prefix("B123.png"))
print(get_prefix("DD500.png"))

A
B
DD


In [ ]:
## finding all unique prefixes
all_prefixes = set()

for label in labels:
    folder = os.path.join(base_dir, label)
    
    for file in os.listdir(folder):
        prefix = get_prefix(file)
        all_prefixes.add(prefix)

all_prefixes = sorted(list(all_prefixes))

print("Total Unique Prefix Groups:", len(all_prefixes))
print(all_prefixes)

Total Unique Prefix Groups: 54
['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'ZA', 'ZB', 'ZC', 'a', 'b', 'c', 'd', 'e', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 'u', 'v', 'w', 'x', 'y', 'za', 'zb', 'zc']


In [21]:
# Convert all prefixes to lowercase person IDs

person_ids = sorted(list(set(prefix.lower() for prefix in all_prefixes)))

print("Total Unique Persons:", len(person_ids))
print(person_ids)

Total Unique Persons: 28
['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'za', 'zb', 'zc']


In [ ]:
## Split Persons into Train / Validation / Test
random.seed(42)
random.shuffle(person_ids)

n = len(person_ids)

train_ids = person_ids[:int(0.7*n)]
val_ids   = person_ids[int(0.7*n):int(0.85*n)]
test_ids  = person_ids[int(0.85*n):]

print("Total Persons =", n)
print("Train Persons =", len(train_ids))
print("Validation Persons =", len(val_ids))
print("Test Persons =", len(test_ids))

Total Persons = 28
Train Persons = 19
Validation Persons = 4
Test Persons = 5


In [ ]:
## Showing Which Persons Are in Each Split
print("TRAIN PERSONS:")
print(sorted(train_ids))

print("\nVALIDATION PERSONS:")
print(sorted(val_ids))

print("\nTEST PERSONS:")
print(sorted(test_ids))

TRAIN PERSONS:
['b', 'c', 'f', 'g', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'v', 'y', 'za', 'zc']

VALIDATION PERSONS:
['e', 'h', 'w', 'zb']

TEST PERSONS:
['a', 'd', 'i', 'u', 'x']


In [ ]:
##Create Fresh Train / Val / Test Folders
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)

for split in ["train", "val", "test"]:
    for label in labels:
        os.makedirs(os.path.join(output_dir, split, label))

print("Fresh split folders created.")

Fresh split folders created.


In [ ]:
## Copy Images Based on Person ID
def get_split(prefix):

    pid = prefix.lower()

    if pid in train_ids:
        return "train"
    
    elif pid in val_ids:
        return "val"
    
    else:
        return "test"


total = 0

for label in labels:

    src_folder = os.path.join(base_dir, label)

    for file in os.listdir(src_folder):

        prefix = get_prefix(file)
        split = get_split(prefix)

        src = os.path.join(src_folder, file)
        dst = os.path.join(output_dir, split, label, file)

        shutil.copy(src, dst)
        total += 1

print("Total Images Copied =", total)

Total Images Copied = 41464


In [ ]:
## Final Image Count
for split in ["train", "val", "test"]:

    print("\n", split.upper())

    for label in labels:

        folder = os.path.join(output_dir, split, label)
        count = len(os.listdir(folder))

        print(label, "=", count)


 TRAIN
drowsy = 13182
non_drowsy = 11109

 VAL
drowsy = 4162
non_drowsy = 3292

 TEST
drowsy = 4813
non_drowsy = 4906


In [27]:
print(set(train_ids) & set(val_ids))
print(set(train_ids) & set(test_ids))
print(set(val_ids) & set(test_ids))

set()
set()
set()
